In [23]:
def run_shared_task_inference(input_csv_path, output_csv_path):
    # 1. Load data
    df = pd.read_csv(input_csv_path)
    ensemble = load_ensemble()
    all_model_logits = []

    # 2. Run each model in the ensemble
    for name, components in ensemble.items():
        print(f"Processing {name}...")
        model = components["model"]
        tokenizer = components["tokenizer"]
        
        # Ensure make_preprocess_fn is available in your namespace
        prep_fn = make_preprocess_fn(tokenizer) 
        ds = Dataset.from_pandas(df).map(prep_fn, batched=False)
        ds.set_format(type='torch', columns=['input_ids', 'attention_mask', 'logic_features'])
        
        dataloader = torch.utils.data.DataLoader(ds, batch_size=16)
        model_logits = []
        
        with torch.no_grad():
            for batch in dataloader:
                batch = {k: v.to(device) for k, v in batch.items()}
                outputs = model(**batch)
                model_logits.append(outputs["logits"].cpu().numpy())
        
        all_model_logits.append(np.concatenate(model_logits, axis=0))

    # 3. Ensemble Averaging (Soft Voting)
    avg_logits = np.mean(all_model_logits, axis=0)
    final_preds = np.argmax(avg_logits, axis=1)
    
    # 4. FORMAT FOR SHARED TASK
    # Create a new DataFrame with ONLY the required column
    submission_df = pd.DataFrame({'prediction': final_preds})
    
    # 5. Save without the index
    submission_df.to_csv(output_csv_path, index=False)
    
    print(f"\n--- Predictions Format for Shared Task ---")
    print(f"File saved to: {output_csv_path}")
    print(f"Total predictions: {len(submission_df)}")
    print(submission_df.head()) # Preview the first few rows

# --- EXECUTION ---
# Change "test_data.csv" to your actual test filename
run_shared_task_inference("training_data/NLI/dev.csv", "shared_task_predictions.csv")

Loading deberta...


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading modernbert...


Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

...

Processing deberta...


Map:   0%|          | 0/6736 [00:00<?, ? examples/s]

Processing modernbert...


Map:   0%|          | 0/6736 [00:00<?, ? examples/s]


--- Predictions Format for Shared Task ---
File saved to: shared_task_predictions.csv
Total predictions: 6736
   prediction
0           0
1           1
2           1
3           0
4           1
